# Remove the Chain of Thought (only for DeepSeek)

In [5]:
import csv

def clean_context_response(text):
    # Split the text by </think> and take the last segment, then strip whitespace
    parts = text.split('</think>')
    cleaned = parts[-1].strip()
    return cleaned

def process_csv(input_file, output_file):
    with open(input_file, 'r', newline='', encoding='utf-8') as infile, \
         open(output_file, 'w', newline='', encoding='utf-8') as outfile:

        reader = csv.DictReader(infile)
        fieldnames = reader.fieldnames
        writer = csv.DictWriter(outfile, fieldnames=fieldnames)
        writer.writeheader()

        for row in reader:
            if 'Context Response' in row:
                original_text = row['Context Response']
                cleaned_text = clean_context_response(original_text)
                row['Context Response'] = cleaned_text
            writer.writerow(row)

# Example usage
input_filename = 'CSV/Refining/R1JanusAllResponses.csv'
output_filename = 'CSV/Refining/R1JanusAllResponsesWithoutCoT.csv'
process_csv(input_filename, output_filename)

# Remove Markdown Symbols

In [6]:
import pandas as pd

def clean_symbols(text):
    if pd.isna(text):
        return text
    return text.replace('#', '').replace('*', '')

def main(input_csv, output_csv):
    # Lire le fichier CSV
    df = pd.read_csv(input_csv)
    
    # Liste des colonnes à nettoyer
    columns_to_clean = [
        'Context Response',
        'Plot 1 Analysis',
        'Plot 2 Analysis',
        'Plot 3 Analysis',
        'Plot 4 Analysis',
        'Plot 5 Analysis',
        'Plot 6 Analysis',
        'Plot 7 Analysis',
        'Plot 8 Analysis',
        'Plot 9 Analysis',
        'Plot 10 Analysis',
        'Plot 11 Analysis',
        'Plot 12 Analysis',
        'Plot 13 Analysis',
        'Plot 14 Analysis'


    ]
    
    # Parcourir chaque colonne et nettoyer les symboles
    for column in columns_to_clean:
        if column in df.columns:
            df[column] = df[column].apply(clean_symbols)
    
    # Enregistrer le fichier CSV nettoyé
    df.to_csv(output_csv, index=False)

if __name__ == "__main__":
    input_csv = 'CSV/Refining/R1JanusAllResponsesWithoutCoT.csv'  # Remplacez par le chemin de votre fichier CSV d'entrée
    output_csv = 'CSV/Refining/WithoutMarkdownR1JANUS.csv'  # Remplacez par le chemin de votre fichier CSV de sortie
    main(input_csv, output_csv)


# BART Paragraphs

In [7]:
from transformers import pipeline
import pandas as pd

def chunk_text(text, tokenizer, max_input_length=1024):
    tokens = tokenizer.encode(text, truncation=False)
    chunks = []
    
    for i in range(0, len(tokens), max_input_length):
        chunk_ids = tokens[i:i + max_input_length]
        chunk_text = tokenizer.decode(chunk_ids, skip_special_tokens=True)
        chunks.append(chunk_text)
    
    return chunks

def summarize_text(text, summarizer, min_summary_length=50, max_summary_length=100, max_input_length=1024):
    chunks = chunk_text(text, summarizer.tokenizer, max_input_length)
    summaries = []
    
    for chunk in chunks:
        summary = summarizer(chunk, min_length=min_summary_length, max_length=max_summary_length, truncation=True)
        summaries.append(summary[0]['summary_text'])
    
    return " ".join(summaries)

def main():
    # Load pre-trained summarization model
    summarizer = pipeline("summarization", model="sshleifer/distilbart-cnn-12-6")

    # Load the CSV file
    file_path = 'CSV/Refining/WithoutMarkdownR1JANUS.csv'
    output_path = 'CSV/Refining/ParagraphsBARTR1JANUS.csv'

    df = pd.read_csv(file_path)

    # Columns
    context_column = 'Context Response'
    plot_columns = [
        'Plot 1 Analysis', 'Plot 2 Analysis', 'Plot 3 Analysis',
        'Plot 4 Analysis', 'Plot 5 Analysis', 'Plot 6 Analysis',
        'Plot 7 Analysis', 'Plot 8 Analysis', 'Plot 9 Analysis',
        'Plot 10 Analysis', 'Plot 11 Analysis', 'Plot 12 Analysis','Plot 13 Analysis',
        'Plot 14 Analysis'
    ]

    summaries = []

    # Summarize each column's text for each row
    for index, row in df.iterrows():
        context_summary = ""
        plot_summaries = []
        
        # Summarize context response
        if pd.notnull(row[context_column]):
            context_summary = summarize_text(row[context_column], summarizer, min_summary_length=200, max_summary_length=350)
        
        # Summarize each plot trend analysis
        for col in plot_columns:
            if pd.notnull(row[col]):
                plot_summary = summarize_text(row[col], summarizer, min_summary_length=50, max_summary_length=100)
                plot_summaries.append((col, plot_summary))

        # Combine summaries into one string
        summary = context_summary + "\n\n"
        for i in range(0, len(plot_summaries), 4):
            batch = plot_summaries[i:i + 4]
            batch_text = "\n".join([summary for col, summary in batch])
            summary += batch_text + "\n\n"
        
        summaries.append(summary.strip())

    # Add summaries to dataframe and save to CSV
    df['Summary (BART) Reduced'] = summaries
    df.to_csv(output_path, index=False)

if __name__ == "__main__":
    main()


Device set to use mps:0


model.safetensors:   0%|          | 0.00/1.22G [00:00<?, ?B/s]

Your max_length is set to 350, but your input_length is only 302. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=151)
Your max_length is set to 100, but your input_length is only 98. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=49)
Your max_length is set to 100, but your input_length is only 79. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=39)
Your max_length is set to 350, but your input_length is only 345. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=172)


# Paragraphs BERT

In [8]:
import torch
from summarizer.bert import Summarizer
from transformers import BertTokenizer
import pandas as pd

def chunk_text(text, tokenizer, max_input_length=512):
    tokens = tokenizer.encode(text, truncation=False, add_special_tokens=False)
    chunks = []
    
    for i in range(0, len(tokens), max_input_length):
        chunk_ids = tokens[i:i + max_input_length]
        chunk_text = tokenizer.decode(chunk_ids)
        chunks.append(chunk_text)
    
    return chunks

def summarize_text(text, summarizer, tokenizer, max_input_length=512, max_summary_length=150):
    chunks = chunk_text(text, tokenizer, max_input_length)
    summaries = []
    
    for chunk in chunks:
        summary = summarizer(chunk, min_length=100, max_length=max_summary_length)
        summaries.append(''.join(summary))
    
    return " ".join(summaries)

def main():
    # Load pre-trained BERT summarization model
    model = Summarizer()
    tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

    # Load the CSV file
    file_path = 'CSV/Refining/ParagraphsBARTR1JANUS.csv'
    df = pd.read_csv(file_path)

    # Columns
    plot_columns = [
        'Plot 1 Analysis', 'Plot 2 Analysis', 'Plot 3 Analysis',
        'Plot 4 Analysis', 'Plot 5 Analysis', 'Plot 6 Analysis',
        'Plot 7 Analysis', 'Plot 8 Analysis', 'Plot 9 Analysis',
        'Plot 10 Analysis', 'Plot 11 Analysis', 'Plot 12 Analysis','Plot 13 Analysis',
        'Plot 14 Analysis'
    ]

    # Create new column for summaries
    df['Summary (BERT) Reduced'] = ''

    # Summarize each column's text for each row
    for index, row in df.iterrows():
        full_summary = ""

        # Summarize context response
        if pd.notnull(row['Context Response']):
            context_summary = summarize_text(row['Context Response'], model, tokenizer)
            full_summary += context_summary + "\n\n"

        # Summarize plot analyses in groups of four
        for i in range(1, len(plot_columns), 4):
            plot_group = plot_columns[i:i+4]
            block_summary = ""
            for col in plot_group:
                if pd.notnull(row[col]):
                    summary = summarize_text(row[col], model, tokenizer)
                    block_summary += summary + " "
            if block_summary:
                full_summary += block_summary.strip() + "\n\n"

        # Remove the last newline characters
        full_summary = full_summary.strip()
        
        # Assign the full summary to the new column
        df.at[index, 'Summary (BERT) Reduced'] = full_summary

    # Save the updated DataFrame to a new CSV file
    output_file_path = 'CSV/Refining/ParagraphBERTR1JANUS.csv'
    df.to_csv(output_file_path, index=False)
    print(f"Summaries have been saved to {output_file_path}")

if __name__ == "__main__":
    main()

Summaries have been saved to CSV/Refining/ParagraphBERTR1JANUS.csv
